# BRIGHT Biology: semantic retrieval

Ce notebook construit un pipeline reproductible de recherche sémantique :
1. charger les exemples BRIGHT ;
2. encoder les chunks avec un modèle Sentence Transformers ;
3. stocker les vecteurs dans LanceDB ;
4. rechercher et évaluer les résultats ;
5. contextualiser les chunks avec un LLM local, si nécessaire.

Les chemins et paramètres principaux sont regroupés dans la cellule de configuration.

## 1. Configuration
Modifiez uniquement cette cellule pour changer le corpus, le modèle ou les sorties.

In [1]:
from huggingface_hub import login

login()

c:\Users\Augnoo\code\lazy_bird\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

DATASET_NAME = "xlangai/BRIGHT"
SUBJECT = "biology"
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
DEVICE = "cuda"
TOP_K = 10
MAX_LOAD_PER_BATCH = 20_000
MAX_BATCH_SIZE = 64

DATA_DIR = Path("data/BRIGHT") / SUBJECT
EMBEDDINGS_PATH = DATA_DIR / "embedded_chunks.csv"
RESULTS_PATH = DATA_DIR / "vanilla_results.csv"
LANCEDB_PATH = DATA_DIR / "vanilladb"
TABLE_NAME = "chunks"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Corpus: {DATASET_NAME}/{SUBJECT}")
print(f"Sorties: {DATA_DIR}")

Corpus: xlangai/BRIGHT/biology
Sorties: data\BRIGHT\biology


## 2. Imports et chargement des données
Les imports sont centralisés afin d'éviter les réinitialisations dispersées du notebook.

In [3]:
import ast
import gc
import re
from collections import defaultdict

import lancedb
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

docs = load_dataset(DATASET_NAME, "long_documents")[SUBJECT]
chunks = load_dataset(DATASET_NAME, "documents")[SUBJECT]
evals = load_dataset(DATASET_NAME, "examples")[SUBJECT]
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

print(f"Documents: {len(docs):,}")
print(f"Chunks: {len(chunks):,}")
print(f"Questions d'évaluation: {len(evals):,}")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3119.67it/s]


Documents: 524
Chunks: 57,359
Questions d'évaluation: 103


## 3. Contrôle rapide du corpus
Cette étape permet de repérer les textes vides ou atypiquement longs avant l'encodage.

In [ ]:
texts = [str(text) for text in chunks["content"] if text is not None]
char_lengths = np.array([len(text) for text in texts], dtype=np.int64)
word_lengths = np.array([len(text.split()) for text in texts], dtype=np.int64)

print(f"Chunks non vides: {len(texts):,}")
print(f"Longueur moyenne: {char_lengths.mean():.0f} caractères / {word_lengths.mean():.1f} mots")
print(f"Longueur maximale: {char_lengths.max():,} caractères")
print(f"Chunks vides: {len(chunks) - len(texts):,}")

## 4. Encodage par lots
Les lots sont regroupés par longueur pour limiter le padding. En cas d'OOM CUDA, le lot est divisé récursivement puis bascule sur CPU pour un texte isolé.

In [ ]:

all_texts = [str(text) for text in chunks["content"]]
lengths = np.array([len(text) for text in all_texts], dtype=np.int64)

def build_dynamic_batches(lengths, max_load, max_batch_size):
    order = np.argsort(-lengths)
    batches, current, current_max = [], [], 0
    for index in order:
        new_max = max(current_max, lengths[index])
        if current and (len(current) + 1) * new_max > max_load:
            batches.append(current)
            current, current_max = [index], lengths[index]
        elif current and len(current) >= max_batch_size:
            batches.append(current)
            current, current_max = [index], lengths[index]
        else:
            current.append(index)
            current_max = new_max
    if current:
        batches.append(current)
    return batches

def encode_with_retry(batch_texts, fallback_device):
    try:
        return model.encode(
            batch_texts,
            batch_size=len(batch_texts),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        if len(batch_texts) == 1:
            original_device = model.device
            model.to(fallback_device)
            vectors = model.encode(batch_texts, batch_size=1, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
            model.to(original_device)
            return vectors
        midpoint = len(batch_texts) // 2
        return np.vstack((
            encode_with_retry(batch_texts[:midpoint], fallback_device),
            encode_with_retry(batch_texts[midpoint:], fallback_device),
        ))

batches = build_dynamic_batches(lengths, MAX_LOAD_PER_BATCH, MAX_BATCH_SIZE)
embedding_dim = model.get_embedding_dimension()
vectors = np.empty((len(all_texts), embedding_dim), dtype=np.float32)
model.eval()

with torch.inference_mode():
    for batch_indices in tqdm(batches, desc="Embedding"):
        batch_texts = [all_texts[index] for index in batch_indices]
        vectors[batch_indices] = encode_with_retry(batch_texts, "cpu")

assert np.isfinite(vectors).all()
print(f"Vecteurs: {vectors.shape}")

## 5. Préparer et persister les chunks
Le `parent_id` est calculé une seule fois et les vecteurs sont conservés dans un CSV réutilisable.

In [ ]:
parent_pattern = re.compile(r"_\d+_\d+\.txt$")
embedded_chunks = []
for index, chunk in enumerate(chunks):
    embedded_chunks.append({
        "id": chunk["id"],
        "parent_id": parent_pattern.sub(".txt", chunk["id"]),
        "content": chunk["content"],
        "vector": vectors[index].tolist(),
    })

embedded_chunks_df = pd.DataFrame(embedded_chunks)
embedded_chunks_df.to_csv(EMBEDDINGS_PATH, index=False, encoding="utf-8")
print(f"Sauvegardé: {EMBEDDINGS_PATH}")

## 6. Index LanceDB et recherche exemple
Cette table contient les chunks, leur document parent et leur représentation vectorielle.

In [4]:
db = lancedb.connect(str(LANCEDB_PATH))

In [ ]:

# table = db.create_table(TABLE_NAME, data=embedded_chunks, mode="overwrite")

# query = "Why do I only breathe out of one nostril?"
# query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
# search_results = table.search(query_vector).limit(TOP_K).select(["id", "parent_id", "content", "_distance"]).to_list()

# pd.DataFrame(search_results)[["id", "_distance"]]

In [5]:
if TABLE_NAME in db.table_names():
    table = db.open_table(TABLE_NAME)
else:
    table = db.create_table(
        TABLE_NAME,
        data=embedded_chunks,
        mode="create",
    )

print(f"Table prête : {TABLE_NAME}")

Table prête : chunks


C:\Users\Augnoo\AppData\Local\Temp\ipykernel_5248\3153336177.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if TABLE_NAME in db.table_names():


## 7. Évaluation
MAP mesure la précision moyenne par question ; `hit_rate` mesure la proportion de réponses gold retrouvées dans le top-k.

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

In [6]:
def parse_ids(value):
    if isinstance(value, (list, tuple, set)):
        return [str(item) for item in value]
    if value is None:
        return []
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (list, tuple, set)):
            return [str(item) for item in parsed]
    except (SyntaxError, ValueError):
        pass
    return re.findall(r"['\"]([^'\"]+)['\"]", str(value))

def average_precision(hits, relevant_count):
    if not relevant_count:
        return 0.0
    found, total = 0, 0.0
    for rank, hit in enumerate(hits, start=1):
        if hit:
            found += 1
            total += found / rank
    return total / relevant_count


### A- RAG vectoriel

In [ ]:
evaluation_rows = []
for example in tqdm(evals, desc="Evaluation"):
    gold_ids = set(parse_ids(example["gold_ids"]))
    vector = model.encode([example["query"]], convert_to_numpy=True, normalize_embeddings=True)[0]
    found_rows = table.search(vector).limit(TOP_K).select(["id"]).to_list()
    hits = [int(row["id"] in gold_ids) for row in found_rows]
    evaluation_rows.append({
        "query": example["query"],
        "average_precision": average_precision(hits, len(gold_ids)),
        "hit_rate": sum(hits) / len(gold_ids) if gold_ids else 0.0,
        "ids_found": [row["id"] for row in found_rows],
        "hits": hits,
    })

results_df = pd.DataFrame(evaluation_rows)
print(f"MAP@{TOP_K}: {results_df['average_precision'].mean():.4f}")
print(f"Hit rate@{TOP_K}: {results_df['hit_rate'].mean():.4f}")
results_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8")

### B- BM25

In [ ]:
table.create_fts_index("content", replace=True)

In [ ]:
BM25 = []
for example in tqdm(evals, desc="BM25 Evaluation"):
    gold_ids = set(parse_ids(example["gold_ids"]))
    query = example["query"]
    found_rows = table.search(query, query_type="fts").limit(TOP_K * 10).select(["id"]).to_list()
    hits = [int(row["id"] in gold_ids) for row in found_rows]
    BM25.append({
        "query": query,
        "average_precision": average_precision(hits, len(gold_ids)),
        "hit_rate": sum(hits) / len(gold_ids) if gold_ids else 0.0,
        "ids_found": [row["id"] for row in found_rows],
        "hits": hits,
    })
    
average_ap = np.mean([result["average_precision"] for result in BM25])
average_hit_rate = np.mean([result["hit_rate"] for result in BM25])

print(f"\n\\nMAP@{TOP_K * 10}: {average_ap:.4f}")
print(f"Hit rate@{TOP_K * 10}: {average_hit_rate:.4f}")

In [ ]:


query = "What is the capital of China?"
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

pairs = [(query, doc) for doc in documents]
scores = model.predict(pairs)
print(scores)
# [  6.4375 -14.375 ]

rankings = model.rank(query, documents)
print(rankings)
# [{'corpus_id': 0, 'score': 6.4375}, {'corpus_id': 1, 'score': -14.375}]


In [ ]:
scores = model.predict([(query, doc) for doc in documents], activation_fn=torch.nn.Sigmoid())

In [ ]:
scores

###  C- Cross-Encoder

In [12]:
import gc, torch

del model
gc.collect()
torch.cuda.empty_cache()  # si le modèle était sur GPU

NameError: name 'model' is not defined

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

In [7]:
evaluation_rows = []
for example in tqdm(evals, desc="Evaluation"):
    gold_ids = set(parse_ids(example["gold_ids"]))
    query = f"{example['query']}\n{example['reasoning']}"
    vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    found_rows = table.search(vector).limit(TOP_K * 10).select(["id"]).to_list()
    hits = [int(row["id"] in gold_ids) for row in found_rows]
    evaluation_rows.append({
        "query": example["query"],
        "average_precision": average_precision(hits, len(gold_ids)),
        "hit_rate": sum(hits) / len(gold_ids) if gold_ids else 0.0,
        "ids_found": [row["id"] for row in found_rows],
        "hits": hits,
    })

results_df = pd.DataFrame(evaluation_rows)
print(f"MAP@{TOP_K}: {results_df['average_precision'].mean():.4f}")
print(f"Hit rate@{TOP_K}: {results_df['hit_rate'].mean():.4f}")
results_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8")

Evaluation: 100%|██████████| 103/103 [00:29<00:00,  3.46it/s]

MAP@10: 0.1748
Hit rate@10: 0.6024


In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("Qwen/Qwen3-Reranker-4B", device=DEVICE)

In [13]:
import ast
import gc

import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import CrossEncoder

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
MAX_SEQ_LEN = 512             # Longueur max (en tokens) d'une paire
MAX_TOKENS_PER_BATCH = 4096   # Borne estimée tokens par micro-batch
MAX_BATCH_SIZE = 64
RERANK_TOP_K = 10
CLEANUP_EVERY_N_BATCHES = 20

# Verrouillage sur CUDA, interdiction explicite d'utiliser le CPU
DEVICE = torch.device("cuda")

# Initialisation du modèle directement en BFloat16 pour optimiser la VRAM
# low_cpu_mem_usage=True empêche la duplication RAM lors du chargement.
model = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device=str(DEVICE),
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "low_cpu_mem_usage": True
    }
)
model.model.eval()

# ----------------------------------------------------------------------------
# Utilitaires
# ----------------------------------------------------------------------------

def parse_ids_found(value):
    if isinstance(value, (list, tuple, set)):
        return [str(item) for item in value]
    if value is None:
        return []
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (list, tuple, set)):
            return [str(item) for item in parsed]
    except (SyntaxError, ValueError):
        pass
    return [str(value)]

def build_dynamic_batches(lengths, max_tokens_per_batch, max_batch_size):
    """
    Crée les batchs en utilisant une heuristique de longueur.
    """
    order = np.argsort(-np.asarray(lengths))
    batches = []
    current_batch = []
    current_max_length = 0

    for index in order:
        new_max_length = max(current_max_length, lengths[index])
        would_exceed_load = current_batch and (
            (len(current_batch) + 1) * new_max_length > max_tokens_per_batch
        )
        would_exceed_size = current_batch and len(current_batch) >= max_batch_size

        if would_exceed_load or would_exceed_size:
            batches.append(current_batch)
            current_batch = [index]
            current_max_length = lengths[index]
        else:
            current_batch.append(index)
            current_max_length = new_max_length

    if current_batch:
        batches.append(current_batch)
    return batches

def score_batch(model, query, batch_candidates, device):
    """
    Tokenisation paresseuse et inférence sur un micro-batch strict.
    """
    batch_contents = [c["content"] for c in batch_candidates]
    
    # Tokenisation uniquement sur le petit batch courant.
    # return_tensors="pt" alloue directement des tenseurs (pas de listes Python)
    encoded = model.tokenizer(
        [query] * len(batch_contents),
        batch_contents,
        truncation="only_second",
        max_length=MAX_SEQ_LEN,
        padding=True,
        return_tensors="pt",
    )

    encoded = {k: v.to(device, non_blocking=False) for k, v in encoded.items()}

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model.model(**encoded)
        
    # Cast en float32 uniquement pour la stabilité numérique de la sigmoïde
    logits = outputs.logits.reshape(-1).float()
    scores = torch.sigmoid(logits)

    result = scores.detach().cpu().tolist()

    # Désallocation explicite pour libérer le graphe CUDA
    del encoded, outputs, logits, scores
    return result

def score_batch_with_retry(model, query, batch_candidates, device):
    """
    Si le batch est trop lourd pour la VRAM, on le coupe en deux (bissection).
    AUCUN offload vers le CPU n'est permis.
    """
    try:
        scores = score_batch(model, query, batch_candidates, device)
        # Retourne une liste de tuples (id_du_candidat, score)
        return [{"id": c["id"], "score": s} for c, s in zip(batch_candidates, scores)]
        
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch_candidates) == 1:
            raise RuntimeError(
                f"OOM persistant sur le GPU avec un seul item. MAX_SEQ_LEN ({MAX_SEQ_LEN}) "
                "est trop grand pour ta VRAM."
            )
        
        # Séparation mathématique du batch en cas d'OOM
        mid = len(batch_candidates) // 2
        left = score_batch_with_retry(model, query, batch_candidates[:mid], device)
        right = score_batch_with_retry(model, query, batch_candidates[mid:], device)
        return left + right


# ----------------------------------------------------------------------------
# Préparation des données & Boucle principale
# ----------------------------------------------------------------------------

chunks_by_id = {
    str(chunk["id"]): str(chunk["content"])
    for chunk in chunks
    if chunk.get("content") is not None
}

reranked_evaluation_rows = []
batches_since_cleanup = 0

with torch.inference_mode():
    for evaluation_row in tqdm(evaluation_rows, desc="Cross-encoder reranking"):
        query = str(evaluation_row["query"])
        ids_found = parse_ids_found(evaluation_row.get("ids_found"))

        candidates = [
            {"id": chunk_id, "content": chunks_by_id[chunk_id]}
            for chunk_id in ids_found
            if chunk_id in chunks_by_id
        ]

        if not candidates:
            updated_row = evaluation_row.copy()
            updated_row["ids_found"] = []
            reranked_evaluation_rows.append(updated_row)
            continue

        # Heuristique : longueur en caractères divisée par 3 pour estimer les tokens.
        # Extrêmement léger en RAM, aucune instanciation d'objets complexes.
        approx_token_lengths = [
            (len(query) + len(c["content"])) // 3 for c in candidates
        ]

        dynamic_batches = build_dynamic_batches(
            approx_token_lengths,
            max_tokens_per_batch=MAX_TOKENS_PER_BATCH,
            max_batch_size=MAX_BATCH_SIZE,
        )

        scored_candidates = []
        for batch_indices in dynamic_batches:
            # On extrait les candidats réels pour ce batch
            batch_candidates = [candidates[i] for i in batch_indices]
            
            results = score_batch_with_retry(model, query, batch_candidates, DEVICE)
            scored_candidates.extend(results)

            batches_since_cleanup += 1
            if batches_since_cleanup >= CLEANUP_EVERY_N_BATCHES:
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                gc.collect()
                batches_since_cleanup = 0

        scored_candidates.sort(key=lambda x: x["score"], reverse=True)

        updated_row = evaluation_row.copy()
        updated_row["ids_found"] = [c["id"] for c in scored_candidates[:RERANK_TOP_K]]
        reranked_evaluation_rows.append(updated_row)

        del candidates, approx_token_lengths, dynamic_batches, scored_candidates

# Nettoyage final
torch.cuda.synchronize()
torch.cuda.empty_cache()
gc.collect()

evaluation_rows = reranked_evaluation_rows

Cross-encoder reranking: 100%|██████████| 103/103 [3:14:54<00:00, 113.54s/it] 


In [14]:
evaluation_rows[0]

{'query': 'Claim in article about why insects are attracted to light\nIn this article they are addressing the reason insects are attracted to light when they say\nHeat radiation as an attractive component is refuted by the effect of LED lighting, which supplies negligible infrared radiation yet still entraps vast numbers of insects.\nI don\'t see why attraction to LEDs shows they\'re not seeking heat. Could they for example be evolutionarily programmed to associate light with heat? So that even though they don\'t encounter heat near/on the LEDs they still "expect" to?',
 'average_precision': 0.2312221231043711,
 'hit_rate': 0.6,
 'ids_found': ['butterflies_stage/homegadsource1gclidC_61_0.txt',
  'insects_attracted_to_light/Phototaxis_0.txt',
  'ants_follow_each_other/ants.htm_10_0.txt',
  'insects_attracted_to_light/Phototaxis_3.txt',
  'dog_eyes_glow/Tapetum_lucidum_1.txt',
  'ants_follow_each_other/ants.htm_11_0.txt',
  'dog_eyes_green/Tapetum_lucidum_1.txt',
  'vision_of_lights/Visi

In [ ]:
evaluation_rows = []
for example in tqdm(evals, desc="Evaluation"):
    gold_ids = set(parse_ids(example["gold_ids"]))
    query = f"{example['query']}\n{example['reasoning']}"
    vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    found_rows = table.search(vector).limit(TOP_K * 10).select(["id"]).to_list()
    hits = [int(row["id"] in gold_ids) for row in found_rows]
    evaluation_rows.append({
        "query": example["query"],
        "average_precision": average_precision(hits, len(gold_ids)),
        "hit_rate": sum(hits) / len(gold_ids) if gold_ids else 0.0,
        "ids_found": [row["id"] for row in found_rows],
        "hits": hits,
    })

results_df = pd.DataFrame(evaluation_rows)
print(f"MAP@{TOP_K}: {results_df['average_precision'].mean():.4f}")
print(f"Hit rate@{TOP_K}: {results_df['hit_rate'].mean():.4f}")
results_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8")

In [18]:
for eval, exemple in zip(evaluation_rows, evals):
    gold_ids = exemple["gold_ids"]
    ids_found = eval["ids_found"]
    hits = [id in gold_ids for id in ids_found]
    eval["hits"] = hits
    eval["average_precision"] = average_precision(hits, len(gold_ids)) if len(gold_ids) > 0 else 0.0
    eval["hit_rate"] = sum(hits) / len(gold_ids) if len(gold_ids) > 0 else 0.0
    eval["gold_ids"] = gold_ids
    
print(f"MAP@10: #{np.mean([row['average_precision'] for row in evaluation_rows]):.4f}")
print(f"Hit rate@10: #{np.mean([row['hit_rate'] for row in evaluation_rows]):.4f}")

MAP@10: #0.2277
Hit rate@10: #0.4391


In [19]:
evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df.to_csv("data/BRIGHT/biology/cross-encoder.csv", index=False, encoding="utf-8")

## 8. Organisation documents/chunks
Cette vue prépare les paires document parent / chunks pour une éventuelle contextualisation.

In [ ]:
chunks_by_parent = defaultdict(list)
for chunk in chunks:
    parent_id = parent_pattern.sub(".txt", chunk["id"])
    chunks_by_parent[parent_id].append(chunk)

document_groups = [
    {"doc": doc, "chunks": chunks_by_parent.get(doc["id"], [])}
    for doc in docs
]
print(f"Documents regroupés: {len(document_groups):,}")

## 9. Contextualisation optionnelle avec un LLM local
Cette section est volontairement désactivée par défaut. Elle reprend les résultats déjà produits et peut être exécutée seulement si le serveur OpenAI-compatible local est disponible sur `127.0.0.1:8080`.

In [ ]:
# Décommentez cette cellule uniquement si le serveur local est lancé.
# from openai import OpenAI
# import os
# from tqdm.auto import tqdm
#
# SYSTEM_PROMPT = """
# Rewrite the target chunk using only facts explicitly present in the full document.
# Preserve every fact and boundary of the target chunk. Return only the rewritten chunk.
# """.strip()
#
# client = OpenAI(base_url="http://127.0.0.1:8080/v1", api_key="none")
# context_path = DATA_DIR / "context_chunks.csv"
# done_ids = set()
# if context_path.exists():
#     done_ids = set(pd.read_csv(context_path)["id"].astype(str))
#
# for group in tqdm(document_groups, desc="Contextualisation"):
#     for chunk in group["chunks"]:
#         if str(chunk["id"]) in done_ids:
#             continue
#         prompt = f"<document>\n{group['doc']['content']}\n</document>\n<chunk>\n{chunk['content']}\n</chunk>"
#         response = client.chat.completions.create(
#             model="local-model",
#             messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
#             temperature=0.1,
#             extra_body={"chat_template_kwargs": {"enable_thinking": False}},
#         )
#         row = {"id": chunk["id"], "parent_id": group["doc"]["id"], "content": chunk["content"], "refined_content": response.choices[0].message.content}
#         pd.DataFrame([row]).to_csv(context_path, mode="a", header=not context_path.exists(), index=False)
#         done_ids.add(str(chunk["id"]))